In [13]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import os


csv_path = r'I:\GEO DATA ANALYSIS\solar_ghana\open_buildings_110m_GHA.csv'

# Central Accra Pilot Bounding Box (WGS84)
min_lon, max_lon = -0.25, 0.00
min_lat, max_lat = 5.50, 5.65

# Output file
output_gpkg = "accra_buildings_google.gpkg"

print("Loading Google Open Buildings Ghana data...")

# Read the CSV
df = pd.read_csv(csv_path)

print(f"Total buildings in Ghana: {len(df):,}")

# Filter to central Accra bounding box using centroid (fast)
mask = (
    (df['longitude'] >= min_lon) & (df['longitude'] <= max_lon) &
    (df['latitude'] >= min_lat) & (df['latitude'] <= max_lat)
)

df_accra = df[mask].copy()
print(f"Buildings in central Accra pilot area: {len(df_accra):,}")

# Convert WKT geometry string to actual geometry objects
print("Converting WKT geometries...")
df_accra['geometry'] = df_accra['geometry'].apply(wkt.loads)

# Create GeoDataFrame
gdf_accra = gpd.GeoDataFrame(df_accra, geometry='geometry', crs="EPSG:4326")

# Add a simple unique ID
gdf_accra['building_id'] = range(1, len(gdf_accra) + 1)

# Save as GeoPackage (best format for later use)
gdf_accra.to_file(output_gpkg, driver="GPKG")


print(f"Saved filtered Accra buildings to: {output_gpkg}")
print(f"CRS: {gdf_accra.crs}")
print(f"Columns: {list(gdf_accra.columns)}")

Loading Google Open Buildings Ghana data...
Total buildings in Ghana: 12,660,784
Buildings in central Accra pilot area: 619,476
Converting WKT geometries...
Saved filtered Accra buildings to: accra_buildings_google.gpkg
CRS: EPSG:4326
Columns: ['latitude', 'longitude', 'area_in_meters', 'confidence', 'geometry', 'full_plus_code', 'building_id']


In [ ]:
# Visualization - Interactive Map of Filtered Accra Buildings

import folium
import geopandas as gpd

# Load the saved GeoPackage
gdf = gpd.read_file("accra_buildings_google.gpkg")

print(f"Visualizing {len(gdf):,} buildings...")

# Create interactive map centered on central Accra
m = folium.Map(location=[5.58, -0.125], zoom_start=14, tiles="CartoDB positron")

# Add buildings as polygons (semi-transparent blue)
folium.GeoJson(
    gdf,
    name="Google Open Buildings",
    style_function=lambda x: {
        'fillColor': '#3388ff',
        'color': '#3388ff',
        'weight': 1,
        'fillOpacity': 0.4
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['building_id', 'area_in_meters', 'confidence'],
        aliases=['Building ID', 'Area (m²)', 'Confidence'],
        localize=True
    )
).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Save map as HTML file
m.save("accra_buildings_map.html")

print("Interactive map saved as 'accra_buildings_map.html'")
print("Open the HTML file in your browser to inspect the buildings visually.")

# Display map 
m

Visualizing 619,476 buildings...
